# Experiment 2 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 2/_analysis/` and renders, in order:

1. Runs per model.
2. Per-defect-type ? model recall from `defect_recall.csv`.
3. One detailed table per model.
4. Heatmap of per-defect-type recall.
5. Offline test-set score scatter from `test_scores.csv`.

The notebook does not read raw experiment output folders or the source dataset tree.


In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 2"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
RUNS_CSV = ANALYSIS_DIR / "runs_cache.csv"
DEFECT_RECALL_CSV = ANALYSIS_DIR / "defect_recall.csv"
SCORES_CSV = ANALYSIS_DIR / "test_scores.csv"

print(f"Runs CSV: {RUNS_CSV}")
print(f"Defect recall CSV: {DEFECT_RECALL_CSV}")
print(f"Scores CSV: {SCORES_CSV}")


In [ ]:
RUN_CACHE_COLUMNS = [
    "experiment", "experiment_dir", "source_format", "raw_model", "model",
    "category", "gpu_name", "auroc", "aupr", "precision", "recall", "f1",
    "accuracy", "val_auroc", "val_aupr", "val_f1", "val_precision",
    "val_recall", "mean_latency_ms", "throughput_fps", "threshold_value",
    "threshold_mode", "train_samples", "val_samples", "test_samples",
    "fit_seconds", "mean_score_ok", "mean_score_ng",
]


def _first_gpu_name(report: dict) -> str | None:
    devices = (
        report.get("hardware", {})
        .get("accelerator", {})
        .get("cuda_devices", [])
    )
    if devices:
        return devices[0].get("name")
    return None


def _model_name(report: dict) -> str | None:
    raw_model = report.get("model", {}).get("name")
    if raw_model is None:
        return None
    return str(raw_model).replace("anomalib_", "")


def _category_from_report(report: dict) -> str | None:
    input_path = report.get("stream", {}).get("input_path")
    if input_path:
        return Path(input_path).name
    return None


def _cache_row_from_offline_report(report_path: Path) -> dict | None:
    run_dir = report_path.parent.parent
    report = json.loads(report_path.read_text(encoding="utf-8"))
    model = _model_name(report)
    category = _category_from_report(report)
    if model not in ALL_MODELS or category is None:
        return None
    threshold = report.get("threshold", {})
    evaluation = report.get("evaluation", {})
    runtime = report.get("runtime", {})
    return {
        "experiment": run_dir.name,
        "experiment_dir": str(run_dir.resolve()),
        "source_format": "offline_colab",
        "raw_model": report.get("model", {}).get("name"),
        "model": model,
        "category": category,
        "gpu_name": _first_gpu_name(report),
        "auroc": report.get("auroc"),
        "aupr": report.get("aupr"),
        "precision": report.get("precision"),
        "recall": report.get("recall"),
        "f1": report.get("f1"),
        "accuracy": report.get("accuracy"),
        "val_auroc": threshold.get("val_auroc"),
        "val_aupr": threshold.get("val_aupr"),
        "val_f1": threshold.get("val_f1"),
        "val_precision": threshold.get("val_precision"),
        "val_recall": threshold.get("val_recall"),
        "mean_latency_ms": report.get("mean_latency_ms"),
        "throughput_fps": report.get("throughput_fps"),
        "threshold_value": report.get("threshold_used") or threshold.get("threshold"),
        "threshold_mode": report.get("threshold_mode") or threshold.get("mode"),
        "train_samples": evaluation.get("train_samples"),
        "val_samples": evaluation.get("val_samples"),
        "test_samples": evaluation.get("test_samples"),
        "fit_seconds": runtime.get("fit_seconds"),
        "mean_score_ok": report.get("mean_score_ok"),
        "mean_score_ng": report.get("mean_score_ng"),
    }


def append_missing_run_cache_rows(exp_root: Path, cache_path: Path) -> int:
    """Append only report_bundle runs that are absent from the run cache CSV."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if cache_path.exists() and cache_path.stat().st_size > 0:
        cached_experiments = pd.read_csv(cache_path, usecols=["experiment"])
        existing = set(cached_experiments["experiment"].dropna().astype(str))
    else:
        existing = set()

    rows = []
    for report_path in sorted(exp_root.glob("*/report_bundle/report.json")):
        if report_path.parent.parent.name in existing:
            continue
        row = _cache_row_from_offline_report(report_path)
        if row is not None:
            rows.append(row)
    if not rows:
        if not cache_path.exists():
            pd.DataFrame(columns=RUN_CACHE_COLUMNS).to_csv(cache_path, index=False)
        print(f"Cache already complete: {cache_path}")
        return 0

    new_df = pd.DataFrame(rows, columns=RUN_CACHE_COLUMNS)
    write_header = not cache_path.exists() or cache_path.stat().st_size == 0
    new_df.to_csv(cache_path, mode="a", header=write_header, index=False)
    print(f"Appended {len(new_df)} missing rows to {cache_path}.")
    print(new_df.groupby("model").size().sort_index().to_string())
    return len(new_df)


append_missing_run_cache_rows(EXP_ROOT, RUNS_CSV)


In [ ]:
df = pd.read_csv(RUNS_CSV)
defect_df = pd.read_csv(DEFECT_RECALL_CSV)
score_df = pd.read_csv(SCORES_CSV, low_memory=False)
normalize_latency(df)
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated run rows across models: {MODELS_PRESENT}")
print(f"Loaded {len(defect_df)} defect-recall rows and {len(score_df)} score rows.")


## 1. Runs per model

Count of model evaluations found under `Experiment 2/jobB_val_defect_V1/`. The Polymer-sheet dataset is a single product, so there is no per-category breakdown.

In [ ]:
runs_per_model = (
    df.groupby("model").size().reindex(ALL_MODELS, fill_value=0).to_frame("runs")
)
runs_per_model.index.name = "model"
runs_per_model

## 2. Per-defect-type recall — defect × model

Recall broken down by ground-truth defect class. Defect types replace the per-category dimension from Experiment 1; the support column on the right shows the number of NG samples of each type present in the test split.

In [ ]:
defect_types = sorted(defect_df["defect_type"].dropna().unique())

recall_matrix = defect_df.pivot_table(
    index="defect_type", columns="model", values="recall", aggfunc="mean"
).reindex(index=defect_types, columns=MODELS_PRESENT)

support_series = defect_df.groupby("defect_type")["support"].max().reindex(defect_types).fillna(0).astype(int)
recall_matrix_display = recall_matrix.copy()
recall_matrix_display["support"] = support_series
recall_matrix_display.index.name = "defect_type"
recall_matrix_display


## 3. Per-model detailed table

One table per model. With a single product, each model produces a single row; columns are the headline run-level metrics. The `train_samples` / `val_samples` / `test_samples` triple is the offline split: the threshold is fit on the validation split and metrics are reported on the held-out test split.

In [ ]:
PER_MODEL_COLS = [
    "threshold_value", "threshold_mode",
    "train_samples", "val_samples", "test_samples",
    "auroc", "aupr", "precision", "recall", "f1", "accuracy",
    "mean_score_ok", "mean_score_ng",
    "mean_latency_ms", "throughput_fps",
]

for model in MODELS_PRESENT:
    sub = (
        df[df["model"] == model]
        .set_index("experiment")[PER_MODEL_COLS]
        .sort_index()
    )
    print(f"\n=== {model} ({len(sub)} runs) ===")
    display(sub)

## 4. Heatmap — per-defect-type recall (defect × model)

In [ ]:
def defect_recall_heatmap(title: str, *, vmin: float = 0.0, vmax: float = 1.0) -> None:
    mat = recall_matrix.reindex(index=defect_types, columns=MODELS_PRESENT)
    fig, ax = plt.subplots(
        figsize=(1.1 * len(MODELS_PRESENT) + 2, 0.55 * len(defect_types) + 1.5)
    )
    masked = np.ma.masked_invalid(mat.values.astype(float))
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="#e5e5e5")
    im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(MODELS_PRESENT)))
    ax.set_xticklabels(MODELS_PRESENT, rotation=45, ha="right")
    ax.set_yticks(range(len(defect_types)))
    ax.set_yticklabels(defect_types)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(
                    j, i, f"{v:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if v < (vmin + vmax) / 2 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    plt.show()


defect_recall_heatmap("Recall \u2014 defect type \u00d7 model")

## 5. Offline test-set scores - OK vs NG per model

Scatter of the per-image anomaly score over the held-out **test split** of each run.

- **Y axis** is the score **min-max normalized to [0, 1] per plot** (the dashed threshold line is normalized with the same min/max so its position is meaningful within the plot).
- **X axis** is a **deterministic random position** in [0, 1] derived from the image path via MD5 hash, so the same image always lands at the same X coordinate across plots.

Green = OK ground truth, red = NG.


In [ ]:
import hashlib
import os
import re

SCATTER_COLS = 3


def _canon_key(path: str) -> str:
    """Canonical filename key shared between jobA and offline_colab formats."""
    name = os.path.basename(str(path))
    name = re.sub(r"\.(jpg|jpeg|png)$", "", name, flags=re.IGNORECASE)
    m = re.match(r"^\d{6}_(.+)$", name)
    if m:
        name = m.group(1)
    return name.replace("__", "_")


def _stable_x(key: str) -> float:
    """Deterministic X in [0, 1] from a canonical key."""
    h = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


n = len(MODELS_PRESENT)
nrows = math.ceil(n / SCATTER_COLS)
fig, axes = plt.subplots(
    nrows, SCATTER_COLS,
    figsize=(SCATTER_COLS * 4.4, nrows * 3.0),
    squeeze=False,
)
fig.suptitle("Offline test-set scores (OK vs NG) per model", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // SCATTER_COLS][i % SCATTER_COLS]
    row = df[df["model"] == model].iloc[0]
    recs = score_df[
        (score_df["experiment"] == row["experiment"])
        & (score_df["model"] == row["model"])
    ].sort_values("sample_idx")
    if recs.empty:
        ax.set_visible(False)
        continue
    scores = recs["score"].to_numpy(dtype=float)
    labels = recs["label"].to_numpy()
    paths = recs["path"].astype(str).to_numpy() if "path" in recs.columns else recs["sample_idx"].astype(str).to_numpy()

    finite = np.isfinite(scores)
    if not finite.any():
        ax.set_visible(False)
        continue
    s_min = float(scores[finite].min())
    s_max = float(scores[finite].max())
    denom = (s_max - s_min) if s_max > s_min else 1.0
    scores_norm = (scores - s_min) / denom

    x_pos = np.fromiter(
        (_stable_x(_canon_key(p)) for p in paths),
        dtype=float,
        count=len(paths),
    )

    ok = labels == 0
    ng = labels == 1
    ax.scatter(x_pos[ok], scores_norm[ok], s=8, c="tab:green", alpha=0.55, label="OK")
    ax.scatter(x_pos[ng], scores_norm[ng], s=8, c="tab:red", alpha=0.55, label="NG")
    thr = row["threshold_value"]
    if pd.notna(thr):
        thr_norm = (float(thr) - s_min) / denom
        ax.axhline(thr_norm, color="black", lw=0.7, ls="--", label=f"thr={thr_norm:.2f}")
    ax.set_title(model, fontsize=10)
    ax.set_xlabel("stable random position", fontsize=8)
    ax.set_ylabel("score (norm 0..1)", fontsize=8)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-0.05, 1.05)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc="best")
    ax.grid(alpha=0.25)

for j in range(n, nrows * SCATTER_COLS):
    axes[j // SCATTER_COLS][j % SCATTER_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
